## Setup
#### Load relevant Python libaries.

In [1]:
import json
import os
from pathlib import Path

from jinja2 import Template

from risk_atlas_nexus.blocks.inference import OllamaInferenceEngine, RITSInferenceEngine
from risk_atlas_nexus.blocks.inference.params import (
    InferenceEngineCredentials,
    OllamaInferenceEngineParams,
    RITSInferenceEngineParams,
)
from risk_atlas_nexus.data import get_templates_path
from risk_atlas_nexus.library import RiskAtlasNexus
from risk_atlas_nexus.blocks.inference.templates import (
    RISK_SEVERITY_INSTRUCTION,
    RISK_SEVERITY_TEMPLATE,
)

/Users/dhaval/Projects/Usage-Governance/risk-atlas-nexus/src/risk_atlas_nexus/toolkit/job_utils.py:2: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


##### Risk Atlas Nexus uses Large Language Models (LLMs) to infer risks dimensions. Therefore requires access to LLMs to inference or call the model. 

**Available Inference Engines**: WML, Ollama, vLLM, RITS. Please follow the [Inference APIs](https://github.com/IBM/risk-atlas-nexus?tab=readme-ov-file#install-for-inference-apis) guide before going ahead.

*Note:* RITS is intended solely for internal IBM use and requires TUNNELALL VPN for access.

In [2]:
# inference_engine = RITSInferenceEngine(
#     model_name_or_path="meta-llama/llama-3-3-70b-instruct",
#     credentials={
#         "api_key": "cbc683b3a1a7c52d2a73008b785d2811",
#         "api_url": "https://inference-3scale-apicast-production.apps.rits.fmaas.res.ibm.com",
#     },
#     parameters=RITSInferenceEngineParams(max_tokens=1000, temperature=0),
# )

inference_engine = OllamaInferenceEngine(
    model_name_or_path="granite3.2:8b",
    credentials=InferenceEngineCredentials(api_url="localhost:11434"),
    parameters=OllamaInferenceEngineParams(
        num_predict=1000, temperature=0, repeat_penalty=1, num_ctx=8192
    ),
)

[2025-05-07 00:48:32:536] - INFO - RiskAtlasNexus - OLLAMA inference engine will execute requests on the server at localhost:11434.
[2025-05-07 00:48:32:595] - INFO - RiskAtlasNexus - Created OLLAMA inference engine.


##### Create an instance of RiskAtlasNexus

*Note: (Optional)* You can specify your own directory in `RiskAtlasNexus(base_dir=<PATH>)` to utilize custom AI ontologies. If left blank, the system will use the provided AI ontologies.

In [3]:
risk_atlas_nexus = RiskAtlasNexus()

[2025-05-07 00:48:32:668] - INFO - RiskAtlasNexus - Created RiskAtlasNexus instance. Base_dir: None


### UseIntent

In [4]:
userIntent = "In a medical chatbot, generative AI can be employed to create a triage system that assesses patients' symptoms and provides immediate, contextually relevant advice based on their medical history and current condition. The chatbot can analyze the patient's input, identify potential medical issues, and offer tailored recommendations or insights to the patient or healthcare provider. This can help streamline the triage process, ensuring that patients receive the appropriate. level of care and attention, and ultimately improving patient outcomes."

#### Collect all required paramters for getting the Risk Severity

In [5]:
domain = risk_atlas_nexus.identify_domain_from_usecases([userIntent], inference_engine)[
    0
]["answer"]
print("Domain: ", domain)

aiTasks = risk_atlas_nexus.identify_ai_tasks_from_usecases(
    [userIntent], inference_engine
)[0]
print("AITasks: ", aiTasks)

CoT = json.loads(
    Path(os.path.join(get_templates_path(), "cot_examples_updated.json")).read_text()
)
response = risk_atlas_nexus.generate_few_shot_output(
    inference_engine, userIntent, CoT[1:]
)
aiUser = response[0]["answer"]
aiSubject = response[1]["answer"]
print("AIUser: ", aiUser)
print("AISubject: ", aiSubject)

Domain:  Healthcare


Inferring with OLLAMA: 100%|██████████| 1/1 [00:10<00:00, 10.32s/it]


AITasks:  ['Text-to-Image', 'Text-to-Speech', 'Text Classification', 'Summarization', 'Question Answering', 'Feature Extraction', 'Image Classification', 'Image Segmentation', 'Image-to-Image', 'Image Feature Extraction', 'Image-Text-to-Text', 'Image-to-Text', 'Keypoint Detection', 'Mask Generation', 'Object Detection', 'Text Generation', 'Token Classification', 'Translation', 'Unconditional Image Generation', 'Video-Text-to-Text', 'Visual Question Answering', 'Zero-Shot Classification', 'Zero-Shot Image Classification']


Inferring with OLLAMA: 100%|██████████| 2/2 [00:07<00:00,  3.57s/it]

AIUser:  Data analysts and supply chain managers in investment banks
AISubject:  The subject is not specified in the provided intent.


#### Create Prompt messages

In [6]:
messages = [
    {"role": "system", "content": Template(RISK_SEVERITY_INSTRUCTION).render()},
    {
        "role": "user",
        "content": Template(RISK_SEVERITY_TEMPLATE).render(
            userIntent=userIntent,
            domain=domain,
            aiTasks=aiTasks,
            aiUser=aiUser,
            aiSubject=aiSubject,
        ),
    },
]

### APPLY Prompt Template

In [7]:
response = inference_engine.chat(
    messages=[messages],
    postprocessors=["json_object"],
    response_format={
        "type": "object",
        "properties": {
            "Description": {"type": "string"},
            "Classification": {
                "type": "array",
                "items": {
                    "enum": [
                        "Unacceptable Risk",
                        "High Risk",
                        "Limited Risk",
                        "Minimal Risk",
                    ]
                },
            },
            "Relevant Text from the EU AI Act": {},
            "Reasoning": {"type": "string"},
        },
        "required": [
            "Description",
            "Classification",
            "Relevant Text from the EU AI Act",
            "Reasoning",
        ],
    },
)[0].prediction

Inferring with OLLAMA: 100%|██████████| 1/1 [00:42<00:00, 42.15s/it]


#### Risk Severity

In [8]:
response

{'Description': "The AI system intended to be used in a medical chatbot that assesses patients' symptoms and provides immediate, contextually relevant advice based on their medical history and current condition.",
 'Classification': ['High Risk'],
 'Relevant Text from the EU AI Act': {'Section': 'Annex III – paragraph 1 – point 1 – point a',
  'Amendment': 'Amendment 712',
  'Text': 'AI systems intended to be used for biometric identification of natural persons, with the exception of those mentioned in Article 5',
  'Explanation': 'The medical chatbot uses AI for biometric identification (symptom analysis), which falls under this category. However, it does not explicitly mention emotion recognition systems, which are excluded in Article 5.'},
 'Reasoning': "The AI system is classified as 'High Risk' because it involves biometric identification of natural persons, which is explicitly mentioned in the EU AI Act as a high-risk application. The amendment 712 excludes certain biometric iden